# LeetCode 1895 — Largest Magic Square bằng Python

Link đề: https://leetcode.com/problems/largest-magic-square/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def largestMagicSquare(self, grid: List[List[int]]) -> int:
        ...
```

Lộ trình:
1. **Vét cạn trực tiếp**: với mọi hình vuông con, cộng lại mọi hàng/cột/đường chéo (`O(m n K⁴)` worst).
2. **Tổng tiền tố theo hàng, cột và hai đường chéo** — mỗi tổng `O(1)` ⇒ kiểm tra một hình vuông `O(k)` ⇒ `O(m n K²)`.
3. **Duyệt `k` giảm dần + dừng ngay khi tìm thấy** và kiểm tra rẻ trước (hai đường chéo), đắt sau.
4. **Dừng sớm trong lúc kiểm tra** + so sánh bằng `all(...)` có ngắn mạch.

## 1. Mục tiêu học

1. Hiểu định nghĩa **hình vuông kỳ diệu**: mọi tổng hàng, tổng cột và **cả hai** tổng đường chéo bằng nhau (giá trị **không** cần phân biệt; mọi ô `1×1` đều là hình vuông kỳ diệu).
2. Xây **bốn loại tổng tiền tố**: theo hàng, theo cột, đường chéo chính `↘`, đường chéo phụ `↙`, và lấy tổng một đoạn trong `O(1)`.
3. Tổ chức tìm kiếm **từ lớn xuống nhỏ** để trả lời ngay khi gặp hình vuông kỳ diệu đầu tiên.
4. Sắp thứ tự kiểm tra để **loại sớm** (đường chéo trước — chỉ 2 tổng).
5. Kiểm chứng bằng vét cạn cộng trực tiếp trên lưới nhỏ ngẫu nhiên, gồm cả lưới có hình vuông kỳ diệu "cấy" sẵn.

## 2. Đề bài & ví dụ

Một **hình vuông kỳ diệu** `k × k` là lưới `k × k` chứa số nguyên sao cho **mọi tổng hàng, mọi tổng cột và cả hai tổng đường chéo** đều bằng nhau. Các số không cần phân biệt. Mọi lưới `1 × 1` đều là hình vuông kỳ diệu.

Cho lưới số nguyên `m × n`, trả về **kích thước** (cạnh `k`) của hình vuông kỳ diệu **lớn nhất** nằm trong lưới (là một hình vuông con liên tiếp).

**Ví dụ 1:** `grid = [[7,1,4,5,6],[2,5,1,6,4],[1,5,4,3,2],[1,2,7,3,4]]` → `3`. Hình vuông `[[5,1,6],[5,4,3],[2,7,3]]`: mọi hàng/cột/chéo đều bằng `12`.

**Ví dụ 2:** `grid = [[5,1,3,1],[9,3,3,1],[1,3,3,8]]` → `2`.

**Ràng buộc:** `1 ≤ m, n ≤ 50`, `1 ≤ grid[i][j] ≤ 10⁶`.

In [1]:
import random
import time
from typing import List

G1 = [[7, 1, 4, 5, 6], [2, 5, 1, 6, 4], [1, 5, 4, 3, 2], [1, 2, 7, 3, 4]]
G2 = [[5, 1, 3, 1], [9, 3, 3, 1], [1, 3, 3, 8]]
examples = [(G1, 3), (G2, 2)]
for g, want in examples:
    print(f"{len(g)}x{len(g[0])} -> kỳ vọng {want}")

4x5 -> kỳ vọng 3
3x4 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Kiểm tra một hình vuông bằng định nghĩa

In [2]:
def is_magic(sq):
    k = len(sq)
    target = sum(sq[0])
    rows = all(sum(row) == target for row in sq)
    cols = all(sum(sq[i][j] for i in range(k)) == target for j in range(k))
    d1 = sum(sq[i][i] for i in range(k)) == target
    d2 = sum(sq[i][k - 1 - i] for i in range(k)) == target
    return rows and cols and d1 and d2

print(is_magic([[5, 1, 6], [5, 4, 3], [2, 7, 3]]), is_magic([[2, 7, 6], [9, 5, 1], [4, 3, 8]]), is_magic([[1, 2], [2, 1]]))

True True False


`[[1,2],[2,1]]`: hàng và cột đều bằng 3, nhưng đường chéo chính `1 + 1 = 2` ⇒ **không** kỳ diệu — đừng quên đường chéo!

### 3.2 Lấy hình vuông con bằng slicing

In [3]:
def sub(grid, r, c, k):
    return [row[c:c + k] for row in grid[r:r + k]]

print(sub(G1, 1, 1, 3))

[[5, 1, 6], [5, 4, 3], [2, 7, 3]]


### 3.3 Tổng tiền tố theo hàng và cột

- `row[i][j + 1] = row[i][j] + g[i][j]` ⇒ tổng `g[i][c..c+k−1] = row[i][c+k] − row[i][c]`.
- `col[i + 1][j] = col[i][j] + g[i][j]` ⇒ tổng `g[r..r+k−1][j] = col[r+k][j] − col[r][j]`.

In [4]:
def row_col_prefix(g):
    m, n = len(g), len(g[0])
    R = [[0] * (n + 1) for _ in range(m)]
    C = [[0] * n for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            R[i][j + 1] = R[i][j] + g[i][j]
            C[i + 1][j] = C[i][j] + g[i][j]
    return R, C

R, C = row_col_prefix(G1)
print("tổng hàng 1, cột 1..3:", R[1][4] - R[1][1], "| tổng cột 2, hàng 1..3:", C[4][2] - C[1][2])

tổng hàng 1, cột 1..3: 12 | tổng cột 2, hàng 1..3: 12


### 3.4 Tổng tiền tố theo đường chéo

- Chéo `↘`: `D[i + 1][j + 1] = D[i][j] + g[i][j]` ⇒ đoạn từ `(r, c)` dài `k`: `D[r+k][c+k] − D[r][c]`.
- Chéo `↙`: `A[i + 1][j] = A[i][j + 1] + g[i][j]` (kích thước `(m+1) × (n+1)`) ⇒ đoạn từ `(r, c+k−1)` xuống trái dài `k` tới `(r+k−1, c)`: `A[r+k][c] − A[r][c+k]`.

In [5]:
def diag_prefix(g):
    m, n = len(g), len(g[0])
    D = [[0] * (n + 1) for _ in range(m + 1)]
    A = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            D[i + 1][j + 1] = D[i][j] + g[i][j]
            A[i + 1][j] = A[i][j + 1] + g[i][j]
    return D, A

D, A = diag_prefix(G1)
r, c, k = 1, 1, 3
print("chéo chính:", D[r + k][c + k] - D[r][c], "= 5+4+3 | chéo phụ:", A[r + k][c] - A[r][c + k], "= 6+4+2")

chéo chính: 12 = 5+4+3 | chéo phụ: 12 = 6+4+2


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1, hình vuông góc trên-trái `(1, 1)`, cạnh 3:

```
5 1 6
5 4 3
2 7 3
```

| loại | tổng |
|---|---|
| hàng | 12, 12, 12 |
| cột | 12, 12, 12 |
| chéo chính 5+4+3 | 12 |
| chéo phụ 6+4+2 | 12 |

Kỳ diệu ✓. Hình vuông `4 × 4` duy nhất (góc `(0,0)` hoặc `(0,1)`) không kỳ diệu ⇒ đáp án `3`.

**Ý tưởng tìm kiếm:** thử `k` từ `min(m, n)` **xuống** 1; với mỗi `k` thử mọi góc trên-trái; gặp hình vuông kỳ diệu đầu tiên ⇒ trả `k`. Luôn có đáp án `≥ 1`.

In [6]:
for r in range(len(G1) - 2):
    for c in range(len(G1[0]) - 2):
        print(f"góc ({r},{c}) cạnh 3: kỳ diệu? {is_magic(sub(G1, r, c, 3))}")

góc (0,0) cạnh 3: kỳ diệu? False
góc (0,1) cạnh 3: kỳ diệu? False
góc (0,2) cạnh 3: kỳ diệu? False
góc (1,0) cạnh 3: kỳ diệu? False
góc (1,1) cạnh 3: kỳ diệu? True
góc (1,2) cạnh 3: kỳ diệu? False


## 5. Cách 1 — Brute Force / baseline: cộng trực tiếp

In [7]:
def largest_magic_brute(grid):
    m, n = len(grid), len(grid[0])
    for k in range(min(m, n), 1, -1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                if is_magic(sub(grid, r, c, k)):
                    return k
    return 1

for g, want in examples:
    print(largest_magic_brute(g), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** số hình vuông cạnh `k` là `(m−k+1)(n−k+1)`, mỗi lần kiểm tra `O(k²)` ⇒ tổng `Σ_k (m−k+1)(n−k+1) k² = O(m n K³)` ⇒ với lưới `50 × 50` (đáp án 1, phải duyệt hết) khoảng `10⁸` phép — chậm trong Python.

**Nút thắt:** tổng hàng/cột/chéo của các hình vuông chồng nhau được cộng lại từ đầu.

In [8]:
random.seed(0)
for s in [15, 25]:
    g = [[random.randint(1, 10**6) for _ in range(s)] for _ in range(s)]   # gần như chắc chắn đáp án 1
    t0 = time.perf_counter(); r = largest_magic_brute(g)
    print(f"{s}x{s}: {r}, {time.perf_counter() - t0:.3f}s")

15x15: 1, 0.002s
25x25: 1, 0.013s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tiền tố 4 hướng):** mọi tổng cần kiểm (k hàng, k cột, 2 chéo) là tổng một **đoạn liên tiếp** trên một hàng/cột/đường chéo ⇒ `O(1)` mỗi tổng với tiền tố ⇒ kiểm tra một hình vuông `O(k)`.

**Insight 2 (thứ tự duyệt):** duyệt `k` giảm dần và **dừng ngay** ở hình vuông kỳ diệu đầu tiên — nếu có hình vuông lớn, ta không cần nhìn các `k` nhỏ.

**Insight 3 (kiểm tra rẻ trước):** so hai đường chéo với nhau trước (2 phép), rồi các hàng, rồi các cột — đa số hình vuông bị loại ngay ở bước đầu.

**Invariant:** khi xét cạnh `k`, mọi cạnh `> k` đã được kiểm và không có hình vuông kỳ diệu nào.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền tố hàng + cột + 2 chéo

In [9]:
def largest_magic_prefix(grid):
    m, n = len(grid), len(grid[0])
    R, C = row_col_prefix(grid)
    D, A = diag_prefix(grid)

    def check(r, c, k):
        target = D[r + k][c + k] - D[r][c]                    # chéo chính
        if A[r + k][c] - A[r][c + k] != target:               # chéo phụ
            return False
        for i in range(r, r + k):
            if R[i][c + k] - R[i][c] != target:
                return False
        for j in range(c, c + k):
            if C[r + k][j] - C[r][j] != target:
                return False
        return True

    for k in range(min(m, n), 1, -1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                if check(r, c, k):
                    return k
    return 1

for g, want in examples:
    print(largest_magic_prefix(g), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** tiền tố `O(m n)`; kiểm tra `O(k)` cho mỗi trong `O(m n)` hình vuông mỗi `k` ⇒ `O(m n K²)` worst ≈ `50·50·Σk ≈ 3·10⁶`. Space `O(m n)`.

### 7.2 Cách 3 — Chỉ tiền tố hàng/cột, đường chéo cộng trực tiếp

Đường chéo chỉ có 2 tổng mỗi hình vuông, cộng trực tiếp `O(k)` cũng không làm tăng bậc. Ít mảng hơn, dễ viết hơn khi phỏng vấn.

In [10]:
def largest_magic_rowcol(grid):
    m, n = len(grid), len(grid[0])
    R, C = row_col_prefix(grid)
    for k in range(min(m, n), 1, -1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                target = R[r][c + k] - R[r][c]
                if any(R[i][c + k] - R[i][c] != target for i in range(r + 1, r + k)):
                    continue
                if any(C[r + k][j] - C[r][j] != target for j in range(c, c + k)):
                    continue
                if sum(grid[r + t][c + t] for t in range(k)) != target:
                    continue
                if sum(grid[r + t][c + k - 1 - t] for t in range(k)) != target:
                    continue
                return k
    return 1

for g, want in examples:
    print(largest_magic_rowcol(g), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** như Cách 2, `O(m n K²)`.

### 7.3 Cách 4 — Lọc nhanh theo tổng toàn hình vuông

Nếu kỳ diệu với tổng mỗi hàng `t`, thì **tổng toàn hình vuông** = `k · t` và `t` = tổng hàng đầu. Dùng **tiền tố 2D** để tính tổng toàn hình vuông `O(1)`, so `== k · t` trước khi kiểm từng hàng/cột — một bộ lọc rẻ loại phần lớn ứng viên.

In [11]:
def largest_magic_filtered(grid):
    m, n = len(grid), len(grid[0])
    R, C = row_col_prefix(grid)
    D, A = diag_prefix(grid)
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = P[i][j + 1] + P[i + 1][j] - P[i][j] + grid[i][j]
    for k in range(min(m, n), 1, -1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                t = D[r + k][c + k] - D[r][c]
                if A[r + k][c] - A[r][c + k] != t:
                    continue
                if P[r + k][c + k] - P[r][c + k] - P[r + k][c] + P[r][c] != k * t:   # lọc rẻ
                    continue
                if all(R[i][c + k] - R[i][c] == t for i in range(r, r + k)) and \
                   all(C[r + k][j] - C[r][j] == t for j in range(c, c + k)):
                    return k
    return 1

for g, want in examples:
    print(largest_magic_filtered(g), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** worst vẫn `O(m n K²)` (khi bộ lọc không loại được), nhưng trung bình gần `O(m n K)` với dữ liệu ngẫu nhiên.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 — in mọi ứng viên bị loại ở bước nào:

In [12]:
def trace_prefix(grid):
    m, n = len(grid), len(grid[0]); R, C = row_col_prefix(grid); D, A = diag_prefix(grid)
    for k in range(min(m, n), 1, -1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                t = D[r + k][c + k] - D[r][c]; t2 = A[r + k][c] - A[r][c + k]
                if t2 != t:
                    print(f"k={k} góc ({r},{c}): chéo chính {t} != chéo phụ {t2} -> loại"); continue
                rows = [R[i][c + k] - R[i][c] for i in range(r, r + k)]
                cols = [C[r + k][j] - C[r][j] for j in range(c, c + k)]
                if any(x != t for x in rows):
                    print(f"k={k} góc ({r},{c}): chéo = {t}, hàng {rows} -> loại"); continue
                if any(x != t for x in cols):
                    print(f"k={k} góc ({r},{c}): chéo = {t}, hàng ok, cột {cols} -> loại"); continue
                print(f"k={k} góc ({r},{c}): mọi tổng = {t} -> KỲ DIỆU, trả về {k}"); return k
    return 1

trace_prefix(G2)

k=3 góc (0,0): chéo chính 11 != chéo phụ 7 -> loại
k=3 góc (0,1): chéo chính 12 != chéo phụ 7 -> loại
k=2 góc (0,0): chéo chính 8 != chéo phụ 10 -> loại
k=2 góc (0,1): chéo chính 4 != chéo phụ 6 -> loại
k=2 góc (0,2): chéo = 4, hàng ok, cột [6, 2] -> loại
k=2 góc (1,0): chéo chính 12 != chéo phụ 4 -> loại
k=2 góc (1,1): mọi tổng = 6 -> KỲ DIỆU, trả về 2


2

In [13]:
trace_prefix(G1)

k=4 góc (0,0): chéo chính 19 != chéo phụ 12 -> loại
k=4 góc (0,1): chéo chính 9 != chéo phụ 18 -> loại
k=3 góc (0,0): chéo chính 16 != chéo phụ 10 -> loại
k=3 góc (0,1): chéo chính 5 != chéo phụ 11 -> loại
k=3 góc (0,2): chéo chính 12 != chéo phụ 16 -> loại
k=3 góc (1,0): chéo chính 14 != chéo phụ 7 -> loại
k=3 góc (1,1): mọi tổng = 12 -> KỲ DIỆU, trả về 3


3

Bảng tóm tắt ví dụ 2 (`3 × 4`): mọi hình vuông `3 × 3` (góc `(0,0)`, `(0,1)`) bị loại; với `k = 2`, góc `(1, 1)`: `[[3,3],[3,3]]` mọi tổng `6` ⇒ đáp án `2`.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def largestMagicSquare(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        R = [[0] * (n + 1) for _ in range(m)]
        C = [[0] * n for _ in range(m + 1)]
        D = [[0] * (n + 1) for _ in range(m + 1)]
        A = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                v = grid[i][j]
                R[i][j + 1] = R[i][j] + v
                C[i + 1][j] = C[i][j] + v
                D[i + 1][j + 1] = D[i][j] + v
                A[i + 1][j] = A[i][j + 1] + v
        for k in range(min(m, n), 1, -1):
            for r in range(m - k + 1):
                for c in range(n - k + 1):
                    t = D[r + k][c + k] - D[r][c]
                    if A[r + k][c] - A[r][c + k] != t:
                        continue
                    if any(R[i][c + k] - R[i][c] != t for i in range(r, r + k)):
                        continue
                    if any(C[r + k][j] - C[r][j] != t for j in range(c, c + k)):
                        continue
                    return k
        return 1

sol = Solution()
for g, want in examples:
    got = sol.largestMagicSquare(g)
    print(got, "OK" if got == want else "SAI")

3 OK
2 OK


## 10. Vì sao thuật toán đúng

**Đầy đủ.** Mọi hình vuông con được xác định bởi góc trên-trái `(r, c)` và cạnh `k` với `r + k ≤ m`, `c + k ≤ n`. Ba vòng lặp duyệt đúng tập này (theo `k` giảm dần).

**Kiểm tra đúng.** `check` so sánh **tất cả** `2k + 2` tổng (k hàng, k cột, 2 chéo) với cùng một giá trị `t` (tổng chéo chính). Hình vuông kỳ diệu ⇔ mọi tổng bằng nhau ⇔ mọi tổng bằng `t`. Các tổng lấy từ tiền tố đúng bằng tổng đoạn tương ứng (mục 3.3–3.4).

**Trả về lớn nhất.** Vì `k` duyệt giảm dần và trả ngay khi gặp hình vuông kỳ diệu đầu tiên, giá trị trả về là `k` lớn nhất có tồn tại. Nếu không có `k ≥ 2` nào, trả `1` (mọi ô đơn là kỳ diệu).

**Bộ lọc của Cách 4 an toàn:** hình vuông kỳ diệu với tổng hàng `t` có tổng toàn bộ `k · t` (cộng `k` hàng) ⇒ ứng viên không thỏa chắc chắn không kỳ diệu.

## 11. Phân tích Time / Space

Gọi `m, n ≤ 50`, `K = min(m, n)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Cộng trực tiếp | `O(m n K³)` | `O(K²)` | lưới nhỏ |
| 2. Tiền tố 4 hướng | `O(m n K²)` | `O(m n)` | **nộp bài** |
| 3. Tiền tố hàng/cột | `O(m n K²)` | `O(m n)` | ít mảng |
| 4. + lọc tổng 2D | `O(m n K²)` worst, nhanh hơn trung bình | `O(m n)` | dữ liệu ngẫu nhiên |

In [15]:
random.seed(1)
g_rand = [[random.randint(1, 10**6) for _ in range(50)] for _ in range(50)]
g_const = [[7] * 50 for _ in range(50)]                     # toàn bằng nhau -> đáp án 50 ngay lập tức
g_small = [[random.randint(1, 2) for _ in range(50)] for _ in range(50)]   # nhiều tổng trùng -> khó lọc
for name, fn in [("prefix", largest_magic_prefix), ("rowcol", largest_magic_rowcol), ("filtered", largest_magic_filtered),
                 ("submit", Solution().largestMagicSquare)]:
    t0 = time.perf_counter(); a = fn(g_rand); b = fn(g_const); c = fn(g_small)
    print(f"{name:8s}: {a} {b} {c}  {time.perf_counter() - t0:.3f}s")

prefix  : 1 50 4  0.009s
rowcol  : 1 50 4  0.018s
filtered: 1 50 4  0.008s
submit  : 1 50 4  0.008s


## 12. Bộ test

Lưới ngẫu nhiên nhỏ với giá trị nhỏ (nhiều tổng trùng ⇒ dễ có hình vuông kỳ diệu "tình cờ"), và lưới có **cấy** một hình vuông kỳ diệu `3 × 3` cổ điển (Lo Shu) hoặc hình vuông hằng.

In [16]:
fns = [largest_magic_prefix, largest_magic_rowcol, largest_magic_filtered, lambda g: Solution().largestMagicSquare(g)]

for g, want in examples:
    assert largest_magic_brute(g) == want
    for fn in fns:
        assert fn(g) == want

LO_SHU = [[2, 7, 6], [9, 5, 1], [4, 3, 8]]
edge = [
    ([[5]], 1), ([[1, 2, 3]], 1), ([[1], [2]], 1),
    ([[1, 1], [1, 1]], 2), ([[1, 2], [2, 1]], 1),            # hàng/cột bằng nhau nhưng chéo khác
    (LO_SHU, 3), ([[3] * 4 for _ in range(4)], 4),
]
for g, want in edge:
    assert largest_magic_brute(g) == want, g
    for fn in fns:
        assert fn(g) == want, (fn, g)

def plant(m, n, sq, rng):
    g = [[rng.randint(1, 50) for _ in range(n)] for _ in range(m)]
    k = len(sq); r = rng.randint(0, m - k); c = rng.randint(0, n - k)
    for i in range(k):
        for j in range(k):
            g[r + i][c + j] = sq[i][j]
    return g

rng = random.Random(2024)
for _ in range(400):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    if rng.random() < 0.4 and m >= 3 and n >= 3:
        g = plant(m, n, LO_SHU, rng)
    else:
        g = [[rng.randint(1, rng.choice([1, 2, 3, 50])) for _ in range(n)] for _ in range(m)]
    want = largest_magic_brute(g)
    for fn in fns:
        assert fn(g) == want, (fn, g)
print("ví dụ + edge + 400 lưới ngẫu nhiên (có cấy Lo Shu) khớp vét cạn")

ví dụ + edge + 400 lưới ngẫu nhiên (có cấy Lo Shu) khớp vét cạn


In [17]:
# tính chất: hình vuông trả về thực sự kỳ diệu, và không có hình vuông lớn hơn
def find_magic(grid):
    k = Solution().largestMagicSquare(grid); m, n = len(grid), len(grid[0])
    for r in range(m - k + 1):
        for c in range(n - k + 1):
            if is_magic(sub(grid, r, c, k)):
                return k, (r, c)
    return k, None

rng = random.Random(7)
for _ in range(200):
    g = [[rng.randint(1, 2) for _ in range(rng.randint(1, 8))] for _ in range(rng.randint(1, 8))]
    w = min(len(row) for row in g); g = [row[:w] for row in g]
    k, pos = find_magic(g)
    assert pos is not None
    assert not any(is_magic(sub(g, r, c, k + 1)) for r in range(len(g) - k) for c in range(w - k))
print("hình vuông trả về kỳ diệu và tối đại (200 lưới)")

hình vuông trả về kỳ diệu và tối đại (200 lưới)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra đường chéo

In [18]:
def wrong_no_diag(grid):
    m, n = len(grid), len(grid[0]); R, C = row_col_prefix(grid)
    for k in range(min(m, n), 1, -1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                t = R[r][c + k] - R[r][c]
                if all(R[i][c + k] - R[i][c] == t for i in range(r, r + k)) and \
                   all(C[r + k][j] - C[r][j] == t for j in range(c, c + k)):
                    return k                       # SAI: chưa kiểm hai đường chéo
    return 1

print("sai =", wrong_no_diag([[1, 2], [2, 1]]), "| đúng =", Solution().largestMagicSquare([[1, 2], [2, 1]]))

sai = 2 | đúng = 1


**Sửa:** kiểm cả hai đường chéo.

### Lỗi 2 — Chỉ kiểm **một** đường chéo

In [19]:
def wrong_one_diag(grid):
    m, n = len(grid), len(grid[0]); R, C = row_col_prefix(grid); D, A = diag_prefix(grid)
    for k in range(min(m, n), 1, -1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                t = D[r + k][c + k] - D[r][c]
                if all(R[i][c + k] - R[i][c] == t for i in range(r, r + k)) and \
                   all(C[r + k][j] - C[r][j] == t for j in range(c, c + k)):
                    return k                       # SAI: thiếu chéo phụ
    return 1

rng = random.Random(3)
for _ in range(20000):
    k = 3
    g = [[rng.randint(1, 3) for _ in range(k)] for _ in range(k)]
    if wrong_one_diag(g) != Solution().largestMagicSquare(g):
        print("phản ví dụ:", g, "| sai =", wrong_one_diag(g), "| đúng =", Solution().largestMagicSquare(g))
        break

phản ví dụ: [[3, 2, 2], [2, 2, 3], [2, 3, 2]] | sai = 3 | đúng = 1


Phản ví dụ (máy tìm): hàng, cột và chéo chính cùng tổng, nhưng chéo phụ khác. **Sửa:** kiểm **cả hai** đường chéo.

### Lỗi 3 — Duyệt `k` tăng dần và trả về `k` đầu tiên tìm được

In [20]:
def wrong_increasing(grid):
    m, n = len(grid), len(grid[0])
    for k in range(2, min(m, n) + 1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                if is_magic(sub(grid, r, c, k)):
                    return k                       # SAI: trả hình vuông NHỎ nhất (>= 2)
    return 1

print("sai =", wrong_increasing([[2] * 4 for _ in range(4)]), "| đúng =", Solution().largestMagicSquare([[2] * 4 for _ in range(4)]))

sai = 2 | đúng = 4


**Sửa:** duyệt `k` **giảm dần** (hoặc duyệt hết và lấy max).

### Lỗi 4 — Sai chỉ số tiền tố chéo phụ

In [21]:
def wrong_anti_index(grid):
    m, n = len(grid), len(grid[0]); R, C = row_col_prefix(grid); D, A = diag_prefix(grid)
    for k in range(min(m, n), 1, -1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                t = D[r + k][c + k] - D[r][c]
                if A[r + k][c] - A[r][c + k - 1] != t:            # SAI: lệch một cột
                    continue
                if all(R[i][c + k] - R[i][c] == t for i in range(r, r + k)) and \
                   all(C[r + k][j] - C[r][j] == t for j in range(c, c + k)):
                    return k
    return 1

for g, want in examples + [(LO_SHU, 3)]:
    print("sai =", wrong_anti_index(g), "| đúng =", want)

sai = 1 | đúng = 3
sai = 1 | đúng = 2
sai = 3 | đúng = 3


**Sửa:** đoạn chéo phụ từ `(r, c+k−1)` tới `(r+k−1, c)` là `A[r+k][c] − A[r][c+k]`. Luôn kiểm công thức tiền tố trên một ví dụ tay (mục 3.4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Nhiều truy vấn tổng trên **hàng / cột / đường chéo** ⇒ tiền tố riêng cho từng hướng.
- "Lớn nhất thỏa điều kiện" mà điều kiện **không đơn điệu** theo kích thước (hình vuông kỳ diệu `k` không suy ra `k − 1`) ⇒ không nhị phân được; duyệt giảm dần + dừng sớm.
- Kiểm tra rẻ trước (đường chéo, tổng toàn bộ) để loại nhanh.
- Lưới `≤ 50` ⇒ `O(m n K²)` là đủ.
- Họ bài: 840 *Magic Squares In Grid* (chỉ `3 × 3`, số 1..9 phân biệt), 1878 *Rhombus Sums* (tiền tố chéo), 304 *Range Sum Query 2D*.

## 15. Biến thể của bài

1. **840 Magic Squares In Grid:** đếm số hình vuông `3 × 3` kỳ diệu với các số `1..9` **phân biệt** — kiểm thêm tập giá trị.
2. **Trả về vị trí** hình vuông kỳ diệu lớn nhất (hàm `find_magic` ở mục 12).
3. **Đếm mọi hình vuông kỳ diệu** (mọi kích thước ≥ 2).
4. **"Semi-magic"** (chỉ hàng và cột, bỏ đường chéo).

In [22]:
def num_magic_3x3(grid):
    """Biến thể 1 (LeetCode 840)."""
    m, n = len(grid), len(grid[0]); cnt = 0
    for r in range(m - 2):
        for c in range(n - 2):
            sq = sub(grid, r, c, 3)
            if sorted(x for row in sq for x in row) == list(range(1, 10)) and is_magic(sq):
                cnt += 1
    return cnt

def count_all_magic(grid):
    """Biến thể 3."""
    m, n = len(grid), len(grid[0])
    return sum(1 for k in range(2, min(m, n) + 1) for r in range(m - k + 1) for c in range(n - k + 1)
               if is_magic(sub(grid, r, c, k)))

print("840:", num_magic_3x3([[4, 3, 8, 4], [9, 5, 1, 9], [2, 7, 6, 2]]))
print("đếm mọi hình vuông kỳ diệu trong lưới 4x4 toàn 1:", count_all_magic([[1] * 4 for _ in range(4)]), "(9 + 4 + 1)")

840: 1
đếm mọi hình vuông kỳ diệu trong lưới 4x4 toàn 1: 14 (9 + 4 + 1)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cộng trực tiếp | định nghĩa | `O(m n K³)` | `O(K²)` | kiểm chứng |
| 2. Tiền tố 4 hướng | mỗi tổng `O(1)` | `O(m n K²)` | `O(m n)` | **nộp bài** |
| 3. Tiền tố hàng/cột | chéo cộng tay | `O(m n K²)` | `O(m n)` | gọn |
| 4. + lọc tổng 2D | loại nhanh | `O(m n K²)` | `O(m n)` | dữ liệu lớn ngẫu nhiên |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `largestMagicSquare([[2,7,6],[9,5,1],[4,3,8]])` và `largestMagicSquare([[1,2],[3,4]])`?

**Bài 2 (tính tay):** với ví dụ 2, liệt kê mọi hình vuông `2 × 2` và cho biết cái nào kỳ diệu.

**Bài 3 (điền TODO):** hoàn thiện `magic_sum_or_none(grid, r, c, k)` trả về tổng chung nếu hình vuông kỳ diệu, ngược lại `None`, dùng tiền tố.

**Bài 4 (tự viết lại):** viết lời giải **không dừng sớm** trả về danh sách mọi `(r, c, k)` kỳ diệu với `k ≥ 2`, rồi lấy `max` — so sánh tốc độ với bản dừng sớm.

**Bài 5 (mini challenge):** sinh ngẫu nhiên một hình vuông kỳ diệu `k × k` (số nguyên dương, không cần phân biệt) bằng cách cộng các "khối" — gợi ý: hình vuông hằng + hoán vị của Lo Shu nhân hệ số; cấy vào lưới ngẫu nhiên `50 × 50` và kiểm tra lời giải tìm ra đúng `k`. (Biến thể tự đặt.)

In [23]:
def magic_sum_or_none(grid, r, c, k):
    # TODO: dùng R, C, D, A (tiền tố) để kiểm 2k + 2 tổng
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Lo Shu: mọi tổng 15 ⇒ `3`. (2) `[[1,2],[3,4]]`: hàng `3, 7` khác nhau. (3) Đáp án `1`.

**Bài 2:** (1) Có `2 · 3 = 6` hình vuông `2 × 2`. (2) Kiểm bằng `is_magic`. (3) Chỉ góc `(1, 1)` (`[[3,3],[3,3]]`) kỳ diệu.

**Bài 3:** (1) `t` = tổng chéo chính. (2) So chéo phụ, các hàng, các cột. (3) Trả `t` hoặc `None`.

**Bài 4:** (1) Bỏ `return`, thêm vào danh sách. (2) `max(k for ...)` hoặc 1. (3) Đo thời gian với `time.perf_counter`.

**Bài 5:** (1) Tổng của hai hình vuông kỳ diệu cùng cỡ là hình vuông kỳ diệu (tổng tuyến tính). (2) Lo Shu xoay/lật vẫn kỳ diệu. (3) Với `k` tổng quát dùng hình vuông hằng (mọi ô bằng nhau).

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert Solution().largestMagicSquare(LO_SHU) == 3 and Solution().largestMagicSquare([[1, 2], [3, 4]]) == 1

# Bài 2
for r in range(2):
    for c in range(3):
        s2 = sub(G2, r, c, 2)
        print(f"góc ({r},{c}) {s2}: kỳ diệu? {is_magic(s2)}")

# Bài 3
def magic_sum_or_none_ans(grid, r, c, k):
    R, C = row_col_prefix(grid); D, A = diag_prefix(grid)
    t = D[r + k][c + k] - D[r][c]
    if A[r + k][c] - A[r][c + k] != t: return None
    if any(R[i][c + k] - R[i][c] != t for i in range(r, r + k)): return None
    if any(C[r + k][j] - C[r][j] != t for j in range(c, c + k)): return None
    return t
assert magic_sum_or_none_ans(G1, 1, 1, 3) == 12 and magic_sum_or_none_ans(G1, 0, 0, 3) is None

# Bài 4
def all_magic(grid):
    m, n = len(grid), len(grid[0])
    return [(r, c, k) for k in range(2, min(m, n) + 1) for r in range(m - k + 1) for c in range(n - k + 1)
            if magic_sum_or_none_ans(grid, r, c, k) is not None]

for g, want in examples:
    found = all_magic(g)
    assert max([k for _, _, k in found], default=1) == want
print("Bài 1-4: OK; ví dụ 1 có các hình vuông kỳ diệu:", all_magic(G1))

góc (0,0) [[5, 1], [9, 3]]: kỳ diệu? False
góc (0,1) [[1, 3], [3, 3]]: kỳ diệu? False
góc (0,2) [[3, 1], [3, 1]]: kỳ diệu? False
góc (1,0) [[9, 3], [1, 3]]: kỳ diệu? False
góc (1,1) [[3, 3], [3, 3]]: kỳ diệu? True
góc (1,2) [[3, 1], [3, 8]]: kỳ diệu? False
Bài 1-4: OK; ví dụ 1 có các hình vuông kỳ diệu: [(1, 1, 3)]


In [25]:
# Bài 5 — cấy hình vuông kỳ diệu vào lưới lớn
def random_magic(k, rng):
    base = rng.randint(1, 1000)
    sq = [[base] * k for _ in range(k)]
    if k == 3:
        lo = LO_SHU
        for _ in range(rng.randint(0, 3)):                    # xoay 90 độ
            lo = [list(row) for row in zip(*lo[::-1])]
        if rng.random() < 0.5:
            lo = [row[::-1] for row in lo]                    # lật
        mul = rng.randint(1, 50)
        sq = [[sq[i][j] + mul * lo[i][j] for j in range(3)] for i in range(3)]
    return sq

rng = random.Random(9)
for _ in range(30):
    k = rng.choice([3, 3, 4, 5, 7])
    sq = random_magic(k, rng); assert is_magic(sq)
    g = plant(50, 50, sq, rng)
    got = Solution().largestMagicSquare(g)
    assert got >= k and got == largest_magic_filtered(g)
print("Bài 5: lời giải tìm được hình vuông kỳ diệu đã cấy (hoặc lớn hơn) trên 30 lưới 50x50")

Bài 5: lời giải tìm được hình vuông kỳ diệu đã cấy (hoặc lớn hơn) trên 30 lưới 50x50


## 20. Template nhớ nhanh + Checklist

```python
# Tiền tố 4 hướng
R[i][j+1] = R[i][j] + g[i][j]          # hàng:  sum g[i][c..c+k-1] = R[i][c+k] - R[i][c]
C[i+1][j] = C[i][j] + g[i][j]          # cột:   sum g[r..r+k-1][j] = C[r+k][j] - C[r][j]
D[i+1][j+1] = D[i][j] + g[i][j]        # ↘:     D[r+k][c+k] - D[r][c]
A[i+1][j] = A[i][j+1] + g[i][j]        # ↙:     A[r+k][c] - A[r][c+k]
for k in range(min(m, n), 1, -1):      # lớn xuống nhỏ, dừng ở hình vuông đầu tiên
    ...  # kiểm chéo -> hàng -> cột, so với t = tổng chéo chính
return 1
```

**Checklist trước khi submit:**
- [ ] Kiểm **cả hai** đường chéo.
- [ ] Duyệt `k` giảm dần, trả về ngay.
- [ ] Mặc định trả `1` (mọi ô đơn kỳ diệu).
- [ ] Công thức tiền tố chéo phụ đúng chỉ số.
- [ ] Thử lưới 1 hàng / 1 cột và lưới hằng.